# Notebook 02 — Preprocessing

**Maternal & Infant Mortality Risk Prediction Using Data Mining Techniques**

> ⚠️ **SYNTHETIC — generated for academic demonstration. Not real clinical data.**

---

## Steps
1. Load the synthetic dataset from `data/raw/`
2. Inspect for missing values, dtypes and outliers
3. Build the scikit-learn preprocessing Pipeline + ColumnTransformer
4. Stratified 80/20 train–test split
5. Apply SMOTE on the **training set only**
6. Save processed data to `data/processed/`

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

from src.preprocessing import (
    preprocess_data, save_processed,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET
)

print('Imports OK')

In [ ]:
# Load raw synthetic data
df = pd.read_csv(os.path.join('..', 'data', 'raw', 'synthetic_maternal_infant_risk.csv'))
print(f"Shape: {df.shape}")
df.head()

## 1. Data Quality Check

In [ ]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

In [ ]:
# Outlier visualisation — boxplots for numeric features
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, col in enumerate(NUMERIC_FEATURES):
    ax = axes[idx // 4, idx % 4]
    df[col].plot(kind='box', ax=ax)
    ax.set_title(col, fontsize=9)

plt.suptitle('SYNTHETIC — Boxplots Before Preprocessing', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join('..', 'reports', 'figures', 'boxplots_before_preprocessing.png'), dpi=150)
plt.show()

## 2. Apply Preprocessing Pipeline

In [ ]:
# Run the full preprocessing routine:
#   - Encode target (HIGH-RISK=1, LOW-RISK=0)
#   - Stratified 80/20 split
#   - Fit ColumnTransformer (impute → cap outliers → scale / encode)
#   - Apply SMOTE on training set ONLY

X_train, X_test, y_train, y_test, preprocessor, feature_names = preprocess_data(
    df, test_size=0.2, random_state=42, apply_smote=True
)

print(f"\nX_train shape (after SMOTE): {X_train.shape}")
print(f"X_test  shape:                {X_test.shape}")
print(f"y_train balance: {np.bincount(y_train)}  (0=LOW-RISK, 1=HIGH-RISK)")
print(f"y_test  balance: {np.bincount(y_test)}")
print(f"\nFeature count: {len(feature_names)}")
print(f"Features: {feature_names}")

## 3. Why SMOTE Must Be Applied AFTER the Train-Test Split

**SMOTE** (Synthetic Minority Over-sampling Technique) generates new synthetic
samples by interpolating between existing minority-class observations.

If SMOTE is applied **before** the split:

1. **Data leakage** — SMOTE-generated training samples may be created from
   (or be very close to) actual test-set samples, because the technique
   interpolates between nearest neighbours.
2. **Inflated metrics** — The model effectively "sees" the test data during
   training, leading to overly optimistic accuracy, recall, and AUC that will
   NOT generalise to unseen real-world data.
3. **Broken evaluation** — The test set is no longer a true hold-out; it
   cannot provide an honest estimate of production performance.

**Correct approach**: Split first, then SMOTE on the **training fold only**.
The test set remains untouched and reflects real class imbalance.

## 4. Save Processed Data

In [ ]:
# Save train/test splits to data/processed/
save_processed(X_train, X_test, y_train, y_test, feature_names)

In [ ]:
# Verify saved files
train_check = pd.read_csv(os.path.join('..', 'data', 'processed', 'train_processed.csv'))
test_check  = pd.read_csv(os.path.join('..', 'data', 'processed', 'test_processed.csv'))

print(f"Train file: {train_check.shape}")
print(f"Test  file: {test_check.shape}")
print("\nSample from processed training set:")
train_check.head()

## Summary

- Missing values: 0 (synthetic data is complete by construction).
- Outliers capped using IQR × 1.5 method.
- Numeric features: median imputation → IQR capping → StandardScaler.
- Categorical features: most-frequent imputation → OneHotEncoder.
- Stratified 80/20 split preserves class proportions.
- SMOTE applied **only on training set** to balance HIGH-RISK and LOW-RISK classes.
- Processed splits saved to `data/processed/`.

**Next → Notebook 03: Exploratory Data Analysis**